# Learning-Rate Schedules

Wiki reference for [learning-rate schedules](https://ml-viz-ruby.vercel.app/wiki/learning-rate-schedules).

**The idea in one sentence.** A fixed learning rate forces a bad compromise — large enough to
make progress means it **bounces** near the optimum on noisy gradients — so schedules
(**warmup** to start stable, then **cosine/step/exp decay** toward zero) let you move fast early
and **settle** precisely at the end.

We implement the common schedules from scratch and run them on a noisy quadratic, **validate the
schedule shapes and that decaying settles where a constant LR bounces**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## 1 — The schedule family

In [ ]:
def step_decay(t, eta0=0.1, gamma=0.1, step=30):
    return eta0 * gamma**(t // step)

def exp_decay(t, eta0=0.1, lam=0.03):
    return eta0 * np.exp(-lam*t)

def cosine(t, T, eta0=0.1, eta_min=0.0):
    return eta_min + 0.5*(eta0-eta_min)*(1 + np.cos(np.pi*t/T))

def warmup_cosine(t, T, Tw=10, eta0=0.1, eta_min=0.0):
    if t < Tw:
        return eta0 * t / Tw
    return cosine(t-Tw, T-Tw, eta0, eta_min)

def one_cycle(t, T, eta_max=0.3, eta_start=0.03):
    up = int(0.4*T)
    if t < up:
        return eta_start + (eta_max-eta_start)*(t/up)
    return eta_max * (1 + np.cos(np.pi*(t-up)/(T-up)))/2

T = 100
ts = np.arange(T)
plt.figure(figsize=(10, 5))
plt.plot(ts, [step_decay(t) for t in ts], label='step decay')
plt.plot(ts, [exp_decay(t) for t in ts], label='exponential')
plt.plot(ts, [cosine(t, T) for t in ts], label='cosine annealing')
plt.plot(ts, [warmup_cosine(t, T) for t in ts], label='linear warmup + cosine')
plt.plot(ts, [one_cycle(t, T) for t in ts], label='one-cycle')
plt.xlabel('step / epoch'); plt.ylabel('learning rate')
plt.title('Learning rate schedules'); plt.legend(); plt.tight_layout(); plt.show()

### Validate: the schedule shapes

Cosine annealing decays smoothly from $\eta_0$ to $\eta_{\min}$ over $T$ steps; warmup-cosine
ramps **up** linearly for the first $T_w$ steps, then decays. We confirm both shapes.

In [ ]:
print(f'cosine: start {cosine(0, 100):.3f} -> end {cosine(100, 100):.4f}')
print(f'warmup: t=0 {warmup_cosine(0, 100):.3f}, t=5 {warmup_cosine(5, 100):.3f}, t=10 {warmup_cosine(10, 100):.3f}')
assert np.isclose(cosine(0, 100), 0.1) and cosine(100, 100) < 1e-9, 'cosine anneals from eta0 to eta_min'
assert warmup_cosine(0, 100) < warmup_cosine(5, 100) < warmup_cosine(10, 100), 'warmup ramps the LR up linearly, then decays'
print('\n✅ warmup starts stable; cosine decays to ~0 to settle at the end')

## 2 — SGDR warm restarts

Cosine to the floor, then jump back up with progressively longer cycles.

In [ ]:
def sgdr(total, eta0=0.1, T0=10, mult=2):
    lrs = []
    t, Ti = 0, T0
    while len(lrs) < total:
        for i in range(Ti):
            if len(lrs) >= total: break
            lrs.append(cosine(i, Ti, eta0, 0.0))
        Ti *= mult
    return np.array(lrs)

lrs = sgdr(120)
plt.figure(figsize=(10, 3.5))
plt.plot(lrs, color='#6366f1')
plt.xlabel('step'); plt.ylabel('learning rate')
plt.title('SGDR: cosine warm restarts (cycles 10, 20, 40, 80...)')
plt.tight_layout(); plt.show()

## 3 — A schedule beats the best constant rate

Minimize a simple quadratic bowl; compare constant rates against cosine decay.

In [ ]:
# Loss: f(x) = x^2 (grad 2x). Noisy gradient to mimic SGD.
rng = np.random.default_rng(0)
def run(schedule_fn, T=100, x0=5.0, noise=0.5):
    x = x0
    losses = []
    for t in range(T):
        eta = schedule_fn(t)
        g = 2*x + rng.normal(0, noise)   # noisy gradient
        x -= eta * g
        losses.append(x**2)
    return losses

T = 100
for eta_const in [0.05, 0.3]:
    plt.plot(run(lambda t, e=eta_const: e), label=f'constant {eta_const}')
plt.plot(run(lambda t: cosine(t, T, eta0=0.3, eta_min=0.001)), label='cosine 0.3->0', lw=2, color='#34d399')
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss x^2 (log)')
plt.title('Cosine decay: fast early, stable late'); plt.legend(); plt.tight_layout(); plt.show()
print("Large constant rate is fast but noisy at the end; small is stable but slow; cosine gets both.")

### Validate: a decaying schedule converges to a low loss

Run on a noisy quadratic, the cosine schedule reaches a low final loss because its rate shrinks
toward zero as it approaches the optimum. We confirm.

In [ ]:
np.random.seed(0)
final_cos = run(lambda t: cosine(t, T))[-1]
print(f'cosine-schedule final loss: {final_cos:.5f}')
assert final_cos < 0.01, 'the cosine schedule converges to a low loss'
print('\n✅ shrinking the LR near the optimum lets the optimizer settle precisely')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **constant LR** | bounces on noise or is too slow (demo) |
| **no warmup** | a high initial LR can destabilize early training (esp. transformers) |
| **wrong $T$** | cosine tuned to the wrong horizon decays too fast/slow |
| **restarts vs monotone** | SGDR restarts can escape minima but add variance |
| **schedule vs adaptive** | Adam still benefits from a schedule on top |

Demo: a decaying schedule settles where a large constant LR keeps bouncing.

In [ ]:
# Why a constant LR is a bad compromise: on NOISY gradients, a rate large enough to move fast
# keeps BOUNCING around the optimum forever (the noise * eta term never shrinks), so the loss
# stays jittery. A schedule that decays toward 0 lets the same run SETTLE. We compare the
# stability of the final iterates.
np.random.seed(0)
loss_const = run(lambda t: 0.3)          # large constant LR
np.random.seed(0)
loss_cos = run(lambda t: cosine(t, T))   # decaying schedule
std_const = np.std(loss_const[-10:])
std_cos = np.std(loss_cos[-10:])
print(f'final-10 loss std: constant LR {std_const:.4f}  vs cosine {std_cos:.6f}')
assert std_cos < std_const, 'a large constant LR keeps bouncing on noise; decaying to 0 lets it settle'
print('\nDecay resolves the fixed-LR dilemma: move fast early, settle precisely late.')

## ✏️ Your turn

**Task A — Transformer inverse-sqrt schedule:** Implement $\eta_t = d^{-0.5}\min(t^{-0.5},\, t\,T_w^{-1.5})$ and plot it; verify the peak occurs exactly at $t = T_w$.

**Task B — Reduce-on-plateau:** Implement a scheduler that tracks a validation loss and multiplies the LR by 0.5 whenever the loss fails to improve for `patience` steps. Feed it a synthetic loss curve that plateaus and show the LR drops occur at the plateaus.

In [ ]:
def transformer_lr(t, d_model=512, warmup=4000):
    # TODO(you): implement the inverse-sqrt schedule (t starts at 1)
    return ...

ts = np.arange(1, 20000)
lrs = [transformer_lr(t) for t in ts]
if lrs[0] is not None:
    peak = ts[np.argmax(lrs)]
    print(f"Peak LR at step {peak} (warmup=4000)")
    plt.plot(ts, lrs, color='#6366f1'); plt.xlabel('step'); plt.ylabel('lr')
    plt.title('Transformer inverse-sqrt schedule'); plt.tight_layout(); plt.show()

<details><summary>Solution — Task A</summary>

```python
def transformer_lr(t, d_model=512, warmup=4000):
    return d_model**-0.5 * min(t**-0.5, t * warmup**-1.5)
# The two branches are equal at t = warmup, so the peak is exactly there.
```
</details>

## Key takeaways

- **Fixed LR is a compromise:** large bounces on noise, small is slow.
- **Warmup** ramps the LR up to avoid early instability (verified shape).
- **Decay (cosine/step/exp)** shrinks the rate to settle at the optimum (verified).
- **Decaying settles where a constant bounces** on noisy gradients (demo).